In [6]:
import os, ctypes, glob

for so_path in glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib/*.so*"):
    try:
        ctypes.CDLL(so_path, mode=ctypes.RTLD_GLOBAL)
    except Exception:
        pass

import gc, json, time, random, shutil, urllib.request
import numpy as np
import faiss
import torch
from pypdf import PdfReader
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from google.colab import files

os.makedirs("tndrr-ai-ml-assignment/task1_lora_finetuning", exist_ok=True)
os.makedirs("tndrr-ai-ml-assignment/task2_rag_from_scratch", exist_ok=True)
os.makedirs("tndrr-ai-ml-assignment/task3_local_inference_optimization", exist_ok=True)

# ==============================================================================
# TASK 1: QLoRA Fine-Tuning (Qwen2.5-0.5B-Instruct -> Strict JSON Extraction)
# ==============================================================================
print(">>> RUNNING TASK 1: QLoRA Fine-Tuning...")
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REQUIRED_KEYS = {"intent", "title", "date", "time", "duration_mins", "participants", "priority"}
SYSTEM_PROMPT = (
    "Extract scheduling parameters into strict minified JSON with keys: "
    "intent, title, date, time, duration_mins, participants, priority. Output JSON only."
)

TEMPLATES = [
    ("Schedule a {dur}-minute {prio} {title} with {p1} and {p2} on {date} at {time}.", "create_event"),
    ("Set up {title} for {date} starting {time} ({dur} mins) — {prio} priority, invite {p1}.", "create_event"),
    ("Can we book a {title} with {p1}, {p2} next {date} around {time} for {dur} minutes? Mark it {prio}.", "create_event"),
    ("Move our {title} with {p1} to {date} at {time} for {dur} mins, keep priority {prio}.", "reschedule_event"),
    ("Reschedule {title} ({dur}m) with {p1} and {p2} to {date} {time} [{prio}].", "reschedule_event"),
]

HARD_TEST_CASES = [
    {
        "input": "Push the Q4 Budget Review with Neha and Vikram to 2026-11-28 at half past two in the afternoon for an hour and a half, urgent!",
        "output": json.dumps({"intent":"reschedule_event","title":"Q4 Budget Review","date":"2026-11-28","time":"14:30","duration_mins":90,"participants":["Neha","Vikram"],"priority":"high"}, separators=(",", ":"))
    },
    {
        "input": "Cancel tomorrow's Sprint Planning with Aarav on 2026-11-20 at 10:00 (30m) low priority.",
        "output": json.dumps({"intent":"cancel_event","title":"Sprint Planning","date":"2026-11-20","time":"10:00","duration_mins":30,"participants":["Aarav"],"priority":"low"}, separators=(",", ":"))
    },
    {
        "input": "Book Client Onboarding Call with Priya, Rohan, and Divya on 2026-11-25 at 11:15 for 45 mins [medium].",
        "output": json.dumps({"intent":"create_event","title":"Client Onboarding Call","date":"2026-11-25","time":"11:15","duration_mins":45,"participants":["Priya","Rohan","Divya"],"priority":"medium"}, separators=(",", ":"))
    }
]

TITLES = ["Sprint Planning", "API Architecture Sync", "Q4 Budget Review", "ML Pipeline Triage", "Client Onboarding Call", "Security Audit Walkthrough"]
PEOPLE = ["Aarav", "Priya", "Rohan", "Neha", "Vikram", "Ananya", "Karthik", "Divya"]
DATES = ["2026-11-20", "2026-11-21", "2026-11-23", "2026-11-24", "2026-11-25", "2026-11-27"]
TIMES = ["09:30", "10:00", "11:15", "14:00", "15:30", "16:45"]
DURATIONS = [15, 30, 45, 60, 90]
PRIORITIES = ["low", "medium", "high"]

rng = random.Random(42)
records = []
for _ in range(357):
    tmpl, intent = rng.choice(TEMPLATES)
    title = rng.choice(TITLES)
    p1, p2 = rng.sample(PEOPLE, 2)
    date, t_str, dur, prio = rng.choice(DATES), rng.choice(TIMES), rng.choice(DURATIONS), rng.choice(PRIORITIES)
    text = tmpl.format(dur=dur, prio=prio, title=title, p1=p1, p2=p2, date=date, time=t_str)
    participants = [p1] if "{p2}" not in tmpl else [p1, p2]
    target = {"intent": intent, "title": title, "date": date, "time": t_str, "duration_mins": dur, "participants": participants, "priority": prio}
    records.append({"input": text, "output": json.dumps(target, separators=(",", ":"))})

train_records, test_records = records[:300], records[300:] + HARD_TEST_CASES

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_prompt(user_text, target_json=None):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    if target_json is not None:
        msgs.append({"role": "assistant", "content": target_json})
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def eval_t1(model):
    model.eval()
    valid, exact, f_hits, failures = 0, 0, 0, []
    for rec in test_records:
        inputs = tokenizer(format_prompt(rec["input"]), return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gt = json.loads(rec["output"])
        try:
            pred = json.loads(gen)
            if isinstance(pred, dict) and REQUIRED_KEYS.issubset(pred.keys()):
                valid += 1
                if pred == gt: exact += 1
                else: failures.append({"input": rec["input"], "expected": gt, "predicted": pred})
                for k in REQUIRED_KEYS:
                    if pred.get(k) == gt.get(k): f_hits += 1
            else:
                failures.append({"input": rec["input"], "expected": gt, "predicted_raw": gen})
        except Exception:
            failures.append({"input": rec["input"], "expected": gt, "predicted_raw": gen})
    n = len(test_records)
    return {
        "schema_valid_json_pct": round(valid / n * 100, 2),
        "exact_json_match_pct": round(exact / n * 100, 2),
        "field_level_accuracy_pct": round(f_hits / (n * len(REQUIRED_KEYS)) * 100, 2),
        "failures": failures[:3]
    }

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
t1_base = eval_t1(base_model)
print("Task 1 Base Done:", {k: v for k, v in t1_base.items() if k != "failures"})

model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
))

def tok_fn(row):
    f_ids = tokenizer(format_prompt(row["input"], row["output"]), truncation=True, max_length=256)["input_ids"]
    p_len = len(tokenizer(format_prompt(row["input"]), truncation=True, max_length=256)["input_ids"])
    return {"input_ids": f_ids, "attention_mask": [1]*len(f_ids), "labels": [-100]*p_len + f_ids[p_len:]}

train_ds = Dataset.from_list(train_records).map(tok_fn, remove_columns=["input", "output"])
Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./tmp_lora", num_train_epochs=3, per_device_train_batch_size=8,
        gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type="cosine",
        warmup_steps=4, logging_steps=20, save_strategy="no", fp16=True,
        optim="paged_adamw_8bit", report_to="none"
    ),
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
).train()

t1_ft = eval_t1(model)
print("Task 1 Fine-Tuned Done:", {k: v for k, v in t1_ft.items() if k != "failures"})
with open("tndrr-ai-ml-assignment/task1_lora_finetuning/eval_results.json", "w") as f:
    json.dump({"base": t1_base, "fine_tuned": t1_ft}, f, indent=2)

del model, base_model
torch.cuda.empty_cache()
gc.collect()

# ==============================================================================
# TASK 2: From-Scratch RAG (NIST SP 800-63B + sentence-transformers + FAISS)
# ==============================================================================
print("\n>>> RUNNING TASK 2: From-Scratch RAG...")
PDF_URL = "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-63b.pdf"
if not os.path.exists("NIST_SP_800-63B.pdf"):
    urllib.request.urlretrieve(PDF_URL, "NIST_SP_800-63B.pdf")

reader = PdfReader("NIST_SP_800-63B.pdf")
pages = [{"page": i+1, "text": " ".join((p.extract_text() or "").split())} for i, p in enumerate(reader.pages) if (p.extract_text() or "").strip()]

TEST_QUESTIONS = [
    {"id": 1, "question": "What are the three Authenticator Assurance Levels (AALs) defined in NIST SP 800-63B?", "key_phrase": "Authenticator Assurance Level 1"},
    {"id": 2, "question": "What is the minimum character length required for user-chosen memorized secrets (passwords)?", "key_phrase": "8 characters in length"},
    {"id": 3, "question": "Why does NIST SP 800-63B prohibit periodic arbitrary password expiration rules?", "key_phrase": "changed arbitrarily"},
    {"id": 4, "question": "Which out-of-band authentication channel requires checking that the number is not VoIP?", "key_phrase": "VoIP"},
    {"id": 5, "question": "What cryptographic property is mandatory for AAL3 authenticators?", "key_phrase": "verifier impersonation resistant"},
    {"id": 6, "question": "What is the maximum allowed lifetime for an out-of-band authentication secret?", "key_phrase": "at most 10 minutes"},
    {"id": 7, "question": "How must verifiers store memorized secrets at rest?", "key_phrase": "salted and hashed"},
    {"id": 8, "question": "What minimum salt length and iteration count apply to password key derivation?", "key_phrase": "32 bits in length"},
    {"id": 9, "question": "Are composition rules requiring uppercase and symbols recommended for passwords?", "key_phrase": "composition rules"},
    {"id": 10, "question": "What must a verifier check newly chosen passwords against during registration?", "key_phrase": "commonly-used"},
    {"id": 11, "question": "What is the maximum number of consecutive failed authentication attempts before rate-limiting?", "key_phrase": "100 failed attempts"},
    {"id": 12, "question": "What periodic and inactivity reauthentication timeouts apply at AAL2?", "key_phrase": "30 minutes"},
    {"id": 13, "question": "What inactivity reauthentication timeout applies at AAL3?", "key_phrase": "15 minutes"},
    {"id": 14, "question": "Should paste functionality be permitted in password input fields?", "key_phrase": "paste"},
    {"id": 15, "question": "Can biometrics be used as a standalone single-factor authenticator?", "key_phrase": "biometrics"}
]

encoder = SentenceTransformer("all-MiniLM-L6-v2")
t2_out = {}
for name, (cw, ov) in [("chunk_150w", (150, 30)), ("chunk_400w", (400, 80))]:
    chunks = []
    step = cw - ov
    for p in pages:
        words = p["text"].split()
        for s in range(0, len(words), step):
            seg = " ".join(words[s:s+cw])
            if len(seg.split()) >= 25:
                chunks.append({"page": p["page"], "text": seg})
    embs = np.asarray(encoder.encode([c["text"] for c in chunks], normalize_embeddings=True), dtype=np.float32)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)

    hits, mrr, missed = 0, 0.0, []
    for q in TEST_QUESTIONS:
        qv = np.asarray(encoder.encode([q["question"]], normalize_embeddings=True), dtype=np.float32)
        _, idxs = index.search(qv, 5)
        top5 = [chunks[i] for i in idxs[0] if i != -1]
        rank_hit = next((r for r, c in enumerate(top5, 1) if q["key_phrase"].lower() in c["text"].lower()), None)
        if rank_hit:
            hits += 1
            mrr += 1.0 / rank_hit
        else:
            missed.append({"id": q["id"], "question": q["question"], "top1": top5[0]["text"][:200]})
    t2_out[name] = {"num_chunks": len(chunks), "recall@5_pct": round(hits/15*100, 2), "mrr@5": round(mrr/15, 4), "hits": f"{hits}/15", "missed": missed}

with open("tndrr-ai-ml-assignment/task2_rag_from_scratch/benchmark_results.json", "w") as f:
    json.dump(t2_out, f, indent=2)

del encoder
torch.cuda.empty_cache()
gc.collect()

# ==============================================================================
# TASK 3: Local Inference & Quantization Benchmark (FP16 vs Q4_K_M)
# ==============================================================================
print("\n>>> RUNNING TASK 3: Local Inference & Quantization Benchmark...")
REPO_ID = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
MODELS = {"FP16_16bit": "qwen2.5-1.5b-instruct-fp16.gguf", "Q4_K_M_4bit": "qwen2.5-1.5b-instruct-q4_k_m.gguf"}
PROMPTS = [
    "Explain the difference between L1 and L2 regularization in 3 concise sentences.",
    "Write a Python function to merge two sorted lists in O(n + m) time without using sorted().",
    "What is the time and space complexity of Dijkstra's algorithm using a binary min-heap?",
    "Convert the SQL query 'SELECT dept, COUNT(*) FROM emp GROUP BY dept HAVING COUNT(*) > 5' into pandas.",
    "Summarize why cosine similarity is preferred over Euclidean distance for normalized text embeddings.",
    "Debug this Python snippet: `def append_item(x, lst=[]): lst.append(x); return lst`.",
    "Give 3 failure modes of naive fixed-size chunking in Retrieval-Augmented Generation.",
    "Derive the gradient of binary cross-entropy loss with respect to logits z.",
    "Compare TCP and UDP across reliability, ordering, and header overhead in a markdown table.",
    "Write a strict JSON object representing a 3-step CI/CD pipeline for a FastAPI microservice."
]

t3_out = {}
for label, fname in MODELS.items():
    path = hf_hub_download(repo_id=REPO_ID, filename=fname)
    torch.cuda.empty_cache()
    free_before, _ = torch.cuda.mem_get_info()
    llm = Llama(model_path=path, n_ctx=2048, n_gpu_layers=-1, verbose=False)
    free_after, _ = torch.cuda.mem_get_info()
    vram_mb = round((free_before - free_after) / (1024**2), 1)

    tot_tok, tot_sec, outputs = 0, 0.0, []
    for p in PROMPTS:
        t0 = time.perf_counter()
        res = llm.create_chat_completion(messages=[{"role": "user", "content": p}], max_tokens=160, temperature=0.0)
        dt = time.perf_counter() - t0
        ntok = res["usage"]["completion_tokens"]
        tot_tok += ntok
        tot_sec += dt
        outputs.append({"prompt": p, "tokens": ntok, "tps": round(ntok/dt, 2), "response": res["choices"][0]["message"]["content"].strip()})
    t3_out[label] = {"vram_used_mb": vram_mb, "avg_tokens_per_sec": round(tot_tok/tot_sec, 2), "outputs": outputs}
    del llm
    gc.collect()

with open("tndrr-ai-ml-assignment/task3_local_inference_optimization/benchmark_summary.json", "w") as f:
    json.dump(t3_out, f, indent=2)

# ==============================================================================
# FINAL SUMMARY OUTPUT & ZIP DOWNLOAD
# ==============================================================================
shutil.make_archive("tndrr-ai-ml-assignment", "zip", ".", "tndrr-ai-ml-assignment")
print("\n" + "="*60)
print("=== ALL TASKS COMPLETE — COPY THESE EXACT NUMBERS INTO YOUR PDF ===")
print("="*60)
print("TASK 1 BASE:", {k: v for k, v in t1_base.items() if k != "failures"})
print("TASK 1 FINE-TUNED:", {k: v for k, v in t1_ft.items() if k != "failures"})
print("TASK 1 SAMPLE FAILURES:", json.dumps(t1_ft["failures"], indent=2))
print("\nTASK 2 150-WORD CHUNKS:", {k: v for k, v in t2_out["chunk_150w"].items() if k != "missed"})
print("TASK 2 400-WORD CHUNKS:", {k: v for k, v in t2_out["chunk_400w"].items() if k != "missed"})
print("TASK 2 MISSED QUERIES (150w):", json.dumps(t2_out["chunk_150w"]["missed"], indent=2))
print("\nTASK 3 FP16:", {"vram_mb": t3_out["FP16_16bit"]["vram_used_mb"], "avg_tps": t3_out["FP16_16bit"]["avg_tokens_per_sec"]})
print("TASK 3 Q4_K_M:", {"vram_mb": t3_out["Q4_K_M_4bit"]["vram_used_mb"], "avg_tps": t3_out["Q4_K_M_4bit"]["avg_tokens_per_sec"]})

files.download("tndrr-ai-ml-assignment.zip")

>>> RUNNING TASK 1: QLoRA Fine-Tuning...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Task 1 Base Done: {'schema_valid_json_pct': 0.0, 'exact_json_match_pct': 0.0, 'field_level_accuracy_pct': 0.0}


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,0.092169
40,0.000335


Task 1 Fine-Tuned Done: {'schema_valid_json_pct': 100.0, 'exact_json_match_pct': 96.67, 'field_level_accuracy_pct': 98.81}

>>> RUNNING TASK 2: From-Scratch RAG...


HTTPError: HTTP Error 403: Forbidden

In [7]:
import os, ctypes, glob, gc, json, time, shutil, requests
import numpy as np
import faiss
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from google.colab import files

for so_path in glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib/*.so*"):
    try:
        ctypes.CDLL(so_path, mode=ctypes.RTLD_GLOBAL)
    except Exception:
        pass

# ==============================================================================
# TASK 2: From-Scratch RAG (NIST SP 800-63B + sentence-transformers + FAISS)
# ==============================================================================
print(">>> RUNNING TASK 2: From-Scratch RAG...")
PDF_URLS = [
    "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-63b.pdf",
    "https://raw.githubusercontent.com/usnistgov/800-63-3/master/sp800-63b/sec1_intro.md"
]
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/128.0.0.0 Safari/537.36"}

if not os.path.exists("NIST_SP_800-63B.pdf"):
    resp = requests.get(PDF_URLS[0], headers=HEADERS, timeout=30)
    resp.raise_for_status()
    with open("NIST_SP_800-63B.pdf", "wb") as f:
        f.write(resp.content)

reader = PdfReader("NIST_SP_800-63B.pdf")
pages = [{"page": i+1, "text": " ".join((p.extract_text() or "").split())} for i, p in enumerate(reader.pages) if (p.extract_text() or "").strip()]
print(f"Loaded {len(pages)} pages from NIST SP 800-63B.")

TEST_QUESTIONS = [
    {"id": 1, "question": "What are the three Authenticator Assurance Levels (AALs) defined in NIST SP 800-63B?", "key_phrase": "Authenticator Assurance Level 1"},
    {"id": 2, "question": "What is the minimum character length required for user-chosen memorized secrets (passwords)?", "key_phrase": "8 characters in length"},
    {"id": 3, "question": "Why does NIST SP 800-63B prohibit periodic arbitrary password expiration rules?", "key_phrase": "changed arbitrarily"},
    {"id": 4, "question": "Which out-of-band authentication channel requires checking that the number is not VoIP?", "key_phrase": "VoIP"},
    {"id": 5, "question": "What cryptographic property is mandatory for AAL3 authenticators?", "key_phrase": "verifier impersonation resistant"},
    {"id": 6, "question": "What is the maximum allowed lifetime for an out-of-band authentication secret?", "key_phrase": "at most 10 minutes"},
    {"id": 7, "question": "How must verifiers store memorized secrets at rest?", "key_phrase": "salted and hashed"},
    {"id": 8, "question": "What minimum salt length and iteration count apply to password key derivation?", "key_phrase": "32 bits in length"},
    {"id": 9, "question": "Are composition rules requiring uppercase and symbols recommended for passwords?", "key_phrase": "composition rules"},
    {"id": 10, "question": "What must a verifier check newly chosen passwords against during registration?", "key_phrase": "commonly-used"},
    {"id": 11, "question": "What is the maximum number of consecutive failed authentication attempts before rate-limiting?", "key_phrase": "100 failed attempts"},
    {"id": 12, "question": "What periodic and inactivity reauthentication timeouts apply at AAL2?", "key_phrase": "30 minutes"},
    {"id": 13, "question": "What inactivity reauthentication timeout applies at AAL3?", "key_phrase": "15 minutes"},
    {"id": 14, "question": "Should paste functionality be permitted in password input fields?", "key_phrase": "paste"},
    {"id": 15, "question": "Can biometrics be used as a standalone single-factor authenticator?", "key_phrase": "biometrics"}
]

encoder = SentenceTransformer("all-MiniLM-L6-v2")
t2_out = {}
for name, (cw, ov) in [("chunk_150w", (150, 30)), ("chunk_400w", (400, 80))]:
    chunks = []
    step = cw - ov
    for p in pages:
        words = p["text"].split()
        for s in range(0, len(words), step):
            seg = " ".join(words[s:s+cw])
            if len(seg.split()) >= 25:
                chunks.append({"page": p["page"], "text": seg})
    embs = np.asarray(encoder.encode([c["text"] for c in chunks], normalize_embeddings=True), dtype=np.float32)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)

    hits, mrr, missed = 0, 0.0, []
    for q in TEST_QUESTIONS:
        qv = np.asarray(encoder.encode([q["question"]], normalize_embeddings=True), dtype=np.float32)
        _, idxs = index.search(qv, 5)
        top5 = [chunks[i] for i in idxs[0] if i != -1]
        rank_hit = next((r for r, c in enumerate(top5, 1) if q["key_phrase"].lower() in c["text"].lower()), None)
        if rank_hit:
            hits += 1
            mrr += 1.0 / rank_hit
        else:
            missed.append({"id": q["id"], "question": q["question"], "top1": top5[0]["text"][:200]})
    t2_out[name] = {"num_chunks": len(chunks), "recall@5_pct": round(hits/15*100, 2), "mrr@5": round(mrr/15, 4), "hits": f"{hits}/15", "missed": missed}

with open("tndrr-ai-ml-assignment/task2_rag_from_scratch/benchmark_results.json", "w") as f:
    json.dump(t2_out, f, indent=2)

del encoder
torch.cuda.empty_cache()
gc.collect()

# ==============================================================================
# TASK 3: Local Inference & Quantization Benchmark (FP16 vs Q4_K_M)
# ==============================================================================
print("\n>>> RUNNING TASK 3: Local Inference & Quantization Benchmark...")
REPO_ID = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
MODELS = {"FP16_16bit": "qwen2.5-1.5b-instruct-fp16.gguf", "Q4_K_M_4bit": "qwen2.5-1.5b-instruct-q4_k_m.gguf"}
PROMPTS = [
    "Explain the difference between L1 and L2 regularization in 3 concise sentences.",
    "Write a Python function to merge two sorted lists in O(n + m) time without using sorted().",
    "What is the time and space complexity of Dijkstra's algorithm using a binary min-heap?",
    "Convert the SQL query 'SELECT dept, COUNT(*) FROM emp GROUP BY dept HAVING COUNT(*) > 5' into pandas.",
    "Summarize why cosine similarity is preferred over Euclidean distance for normalized text embeddings.",
    "Debug this Python snippet: `def append_item(x, lst=[]): lst.append(x); return lst`.",
    "Give 3 failure modes of naive fixed-size chunking in Retrieval-Augmented Generation.",
    "Derive the gradient of binary cross-entropy loss with respect to logits z.",
    "Compare TCP and UDP across reliability, ordering, and header overhead in a markdown table.",
    "Write a strict JSON object representing a 3-step CI/CD pipeline for a FastAPI microservice."
]

t3_out = {}
for label, fname in MODELS.items():
    path = hf_hub_download(repo_id=REPO_ID, filename=fname)
    torch.cuda.empty_cache()
    free_before, _ = torch.cuda.mem_get_info()
    llm = Llama(model_path=path, n_ctx=2048, n_gpu_layers=-1, verbose=False)
    free_after, _ = torch.cuda.mem_get_info()
    vram_mb = round((free_before - free_after) / (1024**2), 1)

    tot_tok, tot_sec, outputs = 0, 0.0, []
    for p in PROMPTS:
        t0 = time.perf_counter()
        res = llm.create_chat_completion(messages=[{"role": "user", "content": p}], max_tokens=160, temperature=0.0)
        dt = time.perf_counter() - t0
        ntok = res["usage"]["completion_tokens"]
        tot_tok += ntok
        tot_sec += dt
        outputs.append({"prompt": p, "tokens": ntok, "tps": round(ntok/dt, 2), "response": res["choices"][0]["message"]["content"].strip()})
    t3_out[label] = {"vram_used_mb": vram_mb, "avg_tokens_per_sec": round(tot_tok/tot_sec, 2), "outputs": outputs}
    del llm
    gc.collect()

with open("tndrr-ai-ml-assignment/task3_local_inference_optimization/benchmark_summary.json", "w") as f:
    json.dump(t3_out, f, indent=2)

# ==============================================================================
# FINAL SUMMARY OUTPUT & ZIP DOWNLOAD
# ==============================================================================
with open("tndrr-ai-ml-assignment/task1_lora_finetuning/eval_results.json") as f:
    t1_saved = json.load(f)

shutil.make_archive("tndrr-ai-ml-assignment", "zip", ".", "tndrr-ai-ml-assignment")
print("\n" + "="*60)
print("=== ALL TASKS COMPLETE — COPY THESE EXACT NUMBERS INTO YOUR PDF ===")
print("="*60)
print("TASK 1 BASE:", {k: v for k, v in t1_saved["base"].items() if k != "failures"})
print("TASK 1 FINE-TUNED:", {k: v for k, v in t1_saved["fine_tuned"].items() if k != "failures"})
print("TASK 1 SAMPLE FAILURES:", json.dumps(t1_saved["fine_tuned"]["failures"], indent=2))
print("\nTASK 2 150-WORD CHUNKS:", {k: v for k, v in t2_out["chunk_150w"].items() if k != "missed"})
print("TASK 2 400-WORD CHUNKS:", {k: v for k, v in t2_out["chunk_400w"].items() if k != "missed"})
print("TASK 2 MISSED QUERIES (150w):", json.dumps(t2_out["chunk_150w"]["missed"], indent=2))
print("\nTASK 3 FP16:", {"vram_mb": t3_out["FP16_16bit"]["vram_used_mb"], "avg_tps": t3_out["FP16_16bit"]["avg_tokens_per_sec"]})
print("TASK 3 Q4_K_M:", {"vram_mb": t3_out["Q4_K_M_4bit"]["vram_used_mb"], "avg_tps": t3_out["Q4_K_M_4bit"]["avg_tokens_per_sec"]})

files.download("tndrr-ai-ml-assignment.zip")

>>> RUNNING TASK 2: From-Scratch RAG...
Loaded 80 pages from NIST SP 800-63B.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (12 outputs left out)

qwen2.5-1.5b-instruct-q4_k_m.gguf: downloading bytes:           |  0.00B            


=== ALL TASKS COMPLETE — COPY THESE EXACT NUMBERS INTO YOUR PDF ===
TASK 1 BASE: {'schema_valid_json_pct': 0.0, 'exact_json_match_pct': 0.0, 'field_level_accuracy_pct': 0.0}
TASK 1 FINE-TUNED: {'schema_valid_json_pct': 100.0, 'exact_json_match_pct': 96.67, 'field_level_accuracy_pct': 98.81}
TASK 1 SAMPLE FAILURES: [
  {
    "input": "Push the Q4 Budget Review with Neha and Vikram to 2026-11-28 at half past two in the afternoon for an hour and a half, urgent!",
    "expected": {
      "intent": "reschedule_event",
      "title": "Q4 Budget Review",
      "date": "2026-11-28",
      "time": "14:30",
      "duration_mins": 90,
      "participants": [
        "Neha",
        "Vikram"
      ],
      "priority": "high"
    },
    "predicted": {
      "intent": "create_event",
      "title": "Q4 Budget Review",
      "date": "2026-11-28",
      "time": "half past two in the afternoon",
      "duration_mins": 1500,
      "participants": [
        "Neha",
        "Vikram"
      ],
      "prior

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>